# Heligman–Pollard — Male (GitHub Edition)

Notebook ini hanya memuat kode analisis mortalitas **Male**: persiapan data, visualisasi, sensitivitas parameter, estimasi NLS, Monte Carlo, dan selang mortalitas. Output telah dihapus.

**Input:** tempatkan `TA26-Mortality Table.xlsx` di folder kerja yang sama, dengan kolom `Age` dan `GAM71-m`. File input tidak termasuk dalam notebook ini.

**Catatan:** simulasi Monte Carlo dapat memerlukan waktu dan sumber daya komputasi yang sangat besar. Notebook belum dieksekusi ulang setelah penyuntingan untuk GitHub.


# Data Mortalitas GAM 71 — Male


In [ ]:
import pandas as pd
from pathlib import Path

# File input publik: letakkan di direktori kerja notebook.
excel_file_path = Path('TA26-Mortality Table.xlsx')
if not excel_file_path.exists():
    raise FileNotFoundError(f'File input tidak ditemukan: {excel_file_path.resolve()}')
df = pd.read_excel(excel_file_path)
display(df.head())


In [ ]:
df_GAM_71_M = df[['Age', 'GAM71-m']].copy()

In [ ]:
df_GAM_71_M = df_GAM_71_M.rename(columns={
      "Age": "age",
      "GAM71-m":"qx"
})

In [ ]:
df_GAM_71_M = df_GAM_71_M.drop(index=111)
df_GAM_71_M

# Visualisasi dan Statistik Deskriptif — Male


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set font family and size
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Computer Modern Roman'] # Use a list for multiple fonts if the first is not found
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Times New Roman'
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14

plt.figure(figsize=(8, 4))
plt.plot(df_GAM_71_M['age'], df_GAM_71_M['qx'], label='$\mathit{Male}$')

# Highlight specific ages with vertical lines and qx values
ages_to_highlight = [20, 40, 60, 80]
for age in ages_to_highlight:
    if age in df_GAM_71_M['age'].values:
        qx_value_at_age = df_GAM_71_M[df_GAM_71_M['age'] == age]['qx'].values[0]

        # Add vertical line
        plt.axvline(x=age, color='gray', linestyle='--', linewidth=2)

        # Add text box with qx value near the curve
        # Adjust text position and box style for better visibility
        plt.text(age + 1, qx_value_at_age, f'{qx_value_at_age:.4f}',
                 color='black', fontsize=12, bbox=dict(facecolor='white',
                                                       alpha=0.7, edgecolor='black', boxstyle='round,pad=0.5'))

plt.xlabel('Usia (tahun)')
plt.ylabel(r'Probabilitas Kematian (${}^{M}q_x$)')
plt.title('Probabilitas Kematian GAM 71 $\mathit{Male}$')
plt.yscale('log') # Add logarithmic scale to y-axis
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
df_M = df_GAM_71_M.copy()

# Ambil usia 0–110 sesuai GAM 71 pada proposal
df_M = df_M[
    (df_M["age"] >= 0) &
    (df_M["age"] <= 110)
].copy()

# Urutkan usia
df_M = df_M.sort_values("age").reset_index(drop=True)

# Ubah menjadi numpy array
age_M = df_M["age"].to_numpy(dtype=float)
qx_M = df_M["qx"].to_numpy(dtype=float)

In [ ]:
print("Jumlah observasi :", len(age_M))
print("Usia minimum     :", age_M.min())
print("Usia maksimum    :", age_M.max())

print("qx minimum       :", qx_M.min())
print("qx maksimum      :", qx_M.max())

print("Semua age finite :", np.all(np.isfinite(age_M)))
print("Semua qx finite  :", np.all(np.isfinite(qx_M)))

# Analisis Sensitivitas Parameter Heligman–Pollard


In [ ]:
import numpy as np
import pandas as pd

# DataFrame parameter A sampai H seperti pada tabel
param_base = {
    "A": 0.05,
    "B": 0.30,
    "C": 0.40,
    "D": 0.01,
    "E": 2.00,
    "F": 25.00,
    "G": 0.0002,
    "H": 1.10,
}

# Tabel parameter
param_df = pd.DataFrame({
    "Parameter": list(param_base.keys()),
    "Nilai_Baseline": list(param_base.values())
})

display(param_df)

# Usia yang dihitung
age = np.arange(0, 111, 1)
age_safe = np.maximum(age, 1e-9)

# Rumus Term 1: A ^ ((x + B) ^ C)
def term_1(x, A, B, C):
    return A ** ((x + B) ** C)

# Rumus Term 2: D * exp(-E * (ln(x) - ln(F))^2)
def term_2(x, D, E, F):
    x_safe = np.maximum(np.asarray(x, dtype=float), 1e-9)
    return D * np.exp(-E * (np.log(x_safe) - np.log(F)) ** 2)

# Rumus Term 3: (G * H^x) / (1 + G * H^x)
def term_3(x, G, H):
    x_arr = np.asarray(x, dtype=float)
    return (G * (H ** x_arr)) / (1 + G * (H ** x_arr))

# Baseline
baseline = param_base.copy()

# Variasi parameter untuk Term 1
A_up = baseline.copy(); A_up["A"] = baseline["A"] * 1.5
A_down = baseline.copy(); A_down["A"] = baseline["A"] * 0.5
B_up = baseline.copy(); B_up["B"] = baseline["B"] * 10
B_down = baseline.copy(); B_down["B"] = baseline["B"] * 0.005
C_up = baseline.copy(); C_up["C"] = baseline["C"] * 1.2
C_down = baseline.copy(); C_down["C"] = baseline["C"] * 0.8

# Variasi parameter untuk Term 2
D_up = baseline.copy(); D_up["D"] = baseline["D"] * 1.5
D_down = baseline.copy(); D_down["D"] = baseline["D"] * 0.7
E_up = baseline.copy(); E_up["E"] = baseline["E"] * 1.2
E_down = baseline.copy(); E_down["E"] = baseline["E"] * 0.8
F_up = baseline.copy(); F_up["F"] = baseline["F"] * 2
F_down = baseline.copy(); F_down["F"] = baseline["F"] * 0.5

# Variasi parameter untuk Term 3
G_up = baseline.copy(); G_up["G"] = baseline["G"] * 1.2
G_down = baseline.copy(); G_down["G"] = baseline["G"] * 0.8
H_up = baseline.copy(); H_up["H"] = baseline["H"] * 1.01
H_down = baseline.copy(); H_down["H"] = baseline["H"] * 0.99

rows = []
for x in age:
    row = {"Age": x}

    row["Baseline A"] = term_1(x, baseline["A"], baseline["B"], baseline["C"])
    row["Baseline A naik"] = term_1(x, A_up["A"], baseline["B"], baseline["C"])
    row["Baseline A turun"] = term_1(x, A_down["A"], baseline["B"], baseline["C"])
    row["Baseline B"] = term_1(x, baseline["A"], baseline["B"], baseline["C"])
    row["Baseline B naik"] = term_1(x, baseline["A"], B_up["B"], baseline["C"])
    row["Baseline B turun"] = term_1(x, baseline["A"], B_down["B"], baseline["C"])
    row["Baseline C"] = term_1(x, baseline["A"], baseline["B"], baseline["C"])
    row["Baseline C naik"] = term_1(x, baseline["A"], baseline["B"], C_up["C"])
    row["Baseline C turun"] = term_1(x, baseline["A"], baseline["B"], C_down["C"])

    row["Baseline D"] = term_2(x, baseline["D"], baseline["E"], baseline["F"])
    row["Baseline D naik"] = term_2(x, D_up["D"], baseline["E"], baseline["F"])
    row["Baseline D turun"] = term_2(x, D_down["D"], baseline["E"], baseline["F"])
    row["Baseline E"] = term_2(x, baseline["D"], baseline["E"], baseline["F"])
    row["Baseline E naik"] = term_2(x, baseline["D"], E_up["E"], baseline["F"])
    row["Baseline E turun"] = term_2(x, baseline["D"], E_down["E"], baseline["F"])
    row["Baseline F"] = term_2(x, baseline["D"], baseline["E"], baseline["F"])
    row["Baseline F naik"] = term_2(x, baseline["D"], baseline["E"], F_up["F"])
    row["Baseline F turun"] = term_2(x, baseline["D"], baseline["E"], F_down["F"])

    row["Baseline G"] = term_3(x, baseline["G"], baseline["H"])
    row["Baseline G naik"] = term_3(x, G_up["G"], baseline["H"])
    row["Baseline G turun"] = term_3(x, G_down["G"], baseline["H"])
    row["Baseline H"] = term_3(x, baseline["G"], baseline["H"])
    row["Baseline H naik"] = term_3(x, baseline["G"], H_up["H"])
    row["Baseline H turun"] = term_3(x, baseline["G"], H_down["H"])

    row["HP"] = row["Baseline A"] + row["Baseline D"] + row["Baseline G"]
    row["HP A naik"] = row["Baseline A naik"] + row["Baseline D"] + row["Baseline G"]
    row["HP A turun"] = row["Baseline A turun"] + row["Baseline D"] + row["Baseline G"]
    row["HP B naik"] = row["Baseline B naik"] + row["Baseline D"] + row["Baseline G"]
    row["HP B turun"] = row["Baseline B turun"] + row["Baseline D"] + row["Baseline G"]
    row["HP C naik"] = row["Baseline C naik"] + row["Baseline D"] + row["Baseline G"]
    row["HP C turun"] = row["Baseline C turun"] + row["Baseline D"] + row["Baseline G"]
    row["HP D naik"] = row["Baseline A"] + row["Baseline D naik"] + row["Baseline G"]
    row["HP D turun"] = row["Baseline A"] + row["Baseline D turun"] + row["Baseline G"]
    row["HP E naik"] = row["Baseline A"] + row["Baseline E naik"] + row["Baseline G"]
    row["HP E turun"] = row["Baseline A"] + row["Baseline E turun"] + row["Baseline G"]
    row["HP F naik"] = row["Baseline A"] + row["Baseline F naik"] + row["Baseline G"]
    row["HP F turun"] = row["Baseline A"] + row["Baseline F turun"] + row["Baseline G"]
    row["HP G naik"] = row["Baseline A"] + row["Baseline D"] + row["Baseline G naik"]
    row["HP G turun"] = row["Baseline A"] + row["Baseline D"] + row["Baseline G turun"]
    row["HP H naik"] = row["Baseline A"] + row["Baseline D"] + row["Baseline H naik"]
    row["HP H turun"] = row["Baseline A"] + row["Baseline D"] + row["Baseline H turun"]
    rows.append(row)

# DataFrame hasil sensitivitas Term 1, 2, dan 3
df_term_sensitivity = pd.DataFrame(rows)
display(df_term_sensitivity.head(10))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# ============================================================
# 2. DATA CHILD MORTALITY
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'A',
        'naik': 'meningkat 50%',
        'turun': 'menurun 50%'
    },
    {
        'parameter': 'B',
        'naik': 'meningkat 1000%',
        'turun': 'menurun 99,5%'
    },
    {
        'parameter': 'C',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    }
]

# ============================================================
# 3. LAYOUT 3 SUBPLOT
# ============================================================

fig = plt.figure(figsize=(16, 11))

fig.suptitle(
    r'Fase $\mathbfit{Child\ Mortality}$',
    fontsize=30,
    fontweight='bold',
    y=0.98
)
gs = fig.add_gridspec(
    2, 4,
    left=0.09,
    right=0.97,
    bottom=0.08,
    top=0.86,
    hspace=0.55,
    wspace=0.85
)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[1, 1:3])
]

# ============================================================
# 4. GRAFIK SENSITIVITAS PARAMETER A, B, C
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    baseline = df_term_sensitivity[
        f'Baseline {p}'
    ].to_numpy()

    naik = df_term_sensitivity[
        f'Baseline {p} naik'
    ].to_numpy()

    turun = df_term_sensitivity[
        f'Baseline {p} turun'
    ].to_numpy()

    # Kurva dasar
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label=f'Baseline {p}'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul subplot
    ax.set_title(
        f'Sensitivitas Parameter {p}',
        fontsize=30,
        pad=18
    )

    # Label sumbu
    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Ukuran angka sumbu
    ax.tick_params(
        axis='both',
        labelsize=21
    )

    # Hilangkan grid
    ax.grid(False)

    # Legenda
    ax.legend(
        loc='upper right',
        frameon=True,
        fontsize=19
    )

# ============================================================
# 5. SIMPAN DAN TAMPILKAN
# ============================================================

plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT DAN GRAFIK
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# Pilihan: 'HP' atau 'Baseline'
JENIS_GRAFIK = 'HP'

# ============================================================
# 2. DATA SENSITIVITAS PARAMETER A, B, C
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'A',
        'naik': 'meningkat 50%',
        'turun': 'menurun 50%'
    },
    {
        'parameter': 'B',
        'naik': 'meningkat 10x',
        'turun': 'menurun 200x'
    },
    {
        'parameter': 'C',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    }
]

# ============================================================
# 3. LAYOUT 2 BARIS DENGAN 3 SUBPLOT
# ============================================================

fig = plt.figure(figsize=(16, 11))

fig.suptitle(
    r'Analisis Sensitivitas Model Heligman-Pollard pada Fase '
    r'$\mathbfit{Child\ Mortality}$',
    fontsize=30,
    fontfamily='Times New Roman',
    fontweight='bold',
    y=0.98
)

gs = fig.add_gridspec(
    2, 4,
    left=0.09,
    right=0.97,
    bottom=0.08,
    top=0.86,
    hspace=0.55,
    wspace=0.85
)

# Dua subplot di atas, satu di tengah bawah
axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[1, 1:3])
]

# ============================================================
# 4. MEMBUAT GRAFIK SENSITIVITAS A, B, C
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    # Pilih data berdasarkan jenis grafik
    if JENIS_GRAFIK == 'HP':

        baseline = df_term_sensitivity['HP'].to_numpy()

        naik = df_term_sensitivity[
            f'HP {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'HP {p} turun'
        ].to_numpy()

        judul = f'Sensitivitas Parameter {p}'

    elif JENIS_GRAFIK == 'Baseline':

        baseline = df_term_sensitivity[
            f'Baseline {p}'
        ].to_numpy()

        naik = df_term_sensitivity[
            f'Baseline {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'Baseline {p} turun'
        ].to_numpy()

        judul = f'Child Mortality - Parameter {p}'

    else:
        raise ValueError(
            "JENIS_GRAFIK harus 'HP' atau 'Baseline'."
        )

    # Kurva baseline
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label='Baseline'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul dan label sumbu
    ax.set_title(
        judul,
        fontsize=30,
        pad=18
    )

    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    ax.tick_params(
        axis='both',
        labelsize=21
    )

    ax.legend(
        loc='best',
        frameon=True,
        fontsize=19
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Hilangkan grid
    ax.grid(False)

# ============================================================
# 5. TAMPILKAN GRAFIK
# ============================================================

plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# ============================================================
# 2. DATA ACCIDENT HUMP
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'D',
        'naik': 'meningkat 50%',
        'turun': 'menurun 30%'
    },
    {
        'parameter': 'E',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    },
    {
        'parameter': 'F',
        'naik': 'meningkat 100%',
        'turun': 'menurun 50%'
    }
]

# ============================================================
# 3. LAYOUT 3 SUBPLOT
# ============================================================

fig = plt.figure(figsize=(16, 11))

fig.suptitle(
    r'Fase $\mathbfit{Accident\ Hump}$',
    fontsize=30,
    fontfamily='Times New Roman',
    fontweight='bold',
    y=0.98
)

gs = fig.add_gridspec(
    2, 4,
    left=0.09,
    right=0.97,
    bottom=0.08,
    top=0.86,
    hspace=0.55,
    wspace=0.85
)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[1, 1:3])
]

# ============================================================
# 4. GRAFIK SENSITIVITAS PARAMETER D, E, F
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    baseline = df_term_sensitivity[
        f'Baseline {p}'
    ].to_numpy()

    naik = df_term_sensitivity[
        f'Baseline {p} naik'
    ].to_numpy()

    turun = df_term_sensitivity[
        f'Baseline {p} turun'
    ].to_numpy()

    # Kurva dasar
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label=f'Baseline {p}'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul subplot
    ax.set_title(
        f'Sensitivitas Parameter {p}',
        fontsize=30,
        pad=18
    )

    # Label sumbu
    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Ukuran angka sumbu
    ax.tick_params(
        axis='both',
        labelsize=21
    )

    # Hilangkan grid
    ax.grid(False)

    # Legenda
    ax.legend(
        loc='upper right',
        frameon=True,
        fontsize=19
    )

# ============================================================
# 5. SIMPAN DAN TAMPILKAN
# ============================================================

plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT DAN GRAFIK
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# Pilihan: 'HP' atau 'Baseline'
JENIS_GRAFIK = 'HP'

# ============================================================
# 2. DATA SENSITIVITAS PARAMETER D, E, F
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'D',
        'naik': 'meningkat 50%',
        'turun': 'menurun 30%'
    },
    {
        'parameter': 'E',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    },
    {
        'parameter': 'F',
        'naik': 'meningkat 100%',
        'turun': 'menurun 50%'
    }
]

# ============================================================
# 3. LAYOUT 2 BARIS DENGAN 3 SUBPLOT
# ============================================================

fig = plt.figure(figsize=(16, 11))

fig.suptitle(
    r'Analisis Sensitivitas Model Heligman-Pollard pada Fase '
    r'$\mathbfit{Accident\ Hump}$',
    fontsize=30,
    fontfamily='Times New Roman',
    fontweight='bold',
    y=0.98
)

gs = fig.add_gridspec(
    2, 4,
    left=0.09,
    right=0.97,
    bottom=0.08,
    top=0.86,
    hspace=0.55,
    wspace=0.85
)

# Dua subplot di atas, satu di tengah bawah
axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[1, 1:3])
]

# ============================================================
# 4. MEMBUAT GRAFIK SENSITIVITAS D, E, F
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    # Pilih data berdasarkan jenis grafik
    if JENIS_GRAFIK == 'HP':

        baseline = df_term_sensitivity['HP'].to_numpy()

        naik = df_term_sensitivity[
            f'HP {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'HP {p} turun'
        ].to_numpy()

        judul = f'Sensitivitas Parameter {p}'

    elif JENIS_GRAFIK == 'Baseline':

        baseline = df_term_sensitivity[
            f'Baseline {p}'
        ].to_numpy()

        naik = df_term_sensitivity[
            f'Baseline {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'Baseline {p} turun'
        ].to_numpy()

        judul = f'Accident Hump - Parameter {p}'

    else:
        raise ValueError(
            "JENIS_GRAFIK harus 'HP' atau 'Baseline'."
        )

    # Kurva baseline
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label='Baseline'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul dan label sumbu
    ax.set_title(
        judul,
        fontsize=30,
        pad=18
    )

    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    ax.tick_params(
        axis='both',
        labelsize=21
    )

    ax.legend(
        loc='best',
        frameon=True,
        fontsize=19
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Hilangkan grid
    ax.grid(False)

# ============================================================
# 5. TAMPILKAN GRAFIK
# ============================================================

plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# ============================================================
# 2. DATA ADULT MORTALITY
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'G',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    },
    {
        'parameter': 'H',
        'naik': 'meningkat 1%',
        'turun': 'menurun 1%'
    }
]

# ============================================================
# 3. LAYOUT 2 SUBPLOT
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 7)
)

fig.suptitle(
    r'Fase $\mathbfit{Adult\ Mortality}$',
    fontsize=30,
    fontfamily='Times New Roman',
    fontweight='bold',
    y=0.98
)

fig.subplots_adjust(
    left=0.09,
    right=0.97,
    bottom=0.20,
    top=0.80,
    wspace=0.45
)

# ============================================================
# 4. GRAFIK SENSITIVITAS PARAMETER G DAN H
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    baseline = df_term_sensitivity[
        f'Baseline {p}'
    ].to_numpy()

    naik = df_term_sensitivity[
        f'Baseline {p} naik'
    ].to_numpy()

    turun = df_term_sensitivity[
        f'Baseline {p} turun'
    ].to_numpy()

    # Kurva dasar
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label=f'Baseline {p}'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul subplot
    ax.set_title(
        f'Sensitivitas Parameter {p}',
        fontsize=30,
        pad=18
    )

    # Label sumbu
    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Ukuran angka sumbu
    ax.tick_params(
        axis='both',
        labelsize=21
    )

    # Hilangkan grid
    ax.grid(False)

    # Legenda
    ax.legend(
        loc='lower right',
        frameon=True,
        fontsize=19
    )

# ============================================================
# 5. SIMPAN DAN TAMPILKAN
# ============================================================

plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 1. PENGATURAN FONT DAN GRAFIK
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.bfit': 'Times New Roman:bold:italic',
    'font.size': 22,
    'axes.titlesize': 30,
    'axes.labelsize': 26,
    'xtick.labelsize': 21,
    'ytick.labelsize': 21,
    'legend.fontsize': 19,
    'axes.grid': False
})

# Pilihan: 'HP' atau 'Baseline'
JENIS_GRAFIK = 'HP'

# ============================================================
# 2. DATA SENSITIVITAS PARAMETER G DAN H
# ============================================================

x = df_term_sensitivity['Age'].to_numpy()

konfigurasi = [
    {
        'parameter': 'G',
        'naik': 'meningkat 20%',
        'turun': 'menurun 20%'
    },
    {
        'parameter': 'H',
        'naik': 'meningkat 1%',
        'turun': 'menurun 1%'
    }
]

# ============================================================
# 3. LAYOUT 2 SUBPLOT
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 7)
)

fig.suptitle(
    r'Analisis Sensitivitas Model Heligman-Pollard pada Fase '
    r'$\mathbfit{Adult\ Mortality}$',
    fontsize=30,
    fontfamily='Times New Roman',
    fontweight='bold',
    y=0.98
)

fig.subplots_adjust(
    left=0.09,
    right=0.97,
    bottom=0.20,
    top=0.80,
    wspace=0.45
)

# ============================================================
# 4. MEMBUAT GRAFIK SENSITIVITAS G DAN H
# ============================================================

for ax, cfg in zip(axes, konfigurasi):

    p = cfg['parameter']

    # Pilih data berdasarkan jenis grafik
    if JENIS_GRAFIK == 'HP':

        baseline = df_term_sensitivity['HP'].to_numpy()

        naik = df_term_sensitivity[
            f'HP {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'HP {p} turun'
        ].to_numpy()

        judul = f'Sensitivitas Parameter {p}'

    elif JENIS_GRAFIK == 'Baseline':

        baseline = df_term_sensitivity[
            f'Baseline {p}'
        ].to_numpy()

        naik = df_term_sensitivity[
            f'Baseline {p} naik'
        ].to_numpy()

        turun = df_term_sensitivity[
            f'Baseline {p} turun'
        ].to_numpy()

        judul = f'Adult Mortality - Parameter {p}'

    else:
        raise ValueError(
            "JENIS_GRAFIK harus 'HP' atau 'Baseline'."
        )

    # Kurva dasar
    ax.semilogy(
        x,
        baseline,
        color='black',
        linewidth=2.5,
        label='Baseline'
    )

    # Parameter meningkat
    ax.semilogy(
        x,
        naik,
        color='green',
        linestyle='dashdot',
        linewidth=2.5,
        label=f'{p} {cfg["naik"]}'
    )

    # Parameter menurun
    ax.semilogy(
        x,
        turun,
        color='red',
        linestyle='dashed',
        linewidth=2.5,
        label=f'{p} {cfg["turun"]}'
    )

    # Judul subplot
    ax.set_title(
        judul,
        fontsize=30,
        pad=18
    )

    # Label sumbu
    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=26,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian ($q_x$)',
        fontsize=26,
        labelpad=12
    )

    # Batas sumbu
    ax.set_xlim(0, 110)
    ax.set_ylim(1e-4, 1)

    # Ukuran angka sumbu
    ax.tick_params(
        axis='both',
        labelsize=21
    )

    # Hilangkan grid
    ax.grid(False)

    # Legenda
    ax.legend(
        loc='lower right',
        frameon=True,
        fontsize=19
    )

# ============================================================
# 5. TAMPILKAN GRAFIK
# ============================================================

plt.show()


# Estimasi Parameter NLS — Male


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

# ============================================================
# 1. KONFIGURASI
# ============================================================

parameter_names_M = ["A", "B", "C", "D", "E", "F", "G", "H"]

parameter_lower_M = np.array([
    1e-12, 1e-12, 1e-12, 1e-12,
    1e-12, 1.0, 1e-12, 1.000001
], dtype=float)

parameter_upper_M = np.array([
    0.9999, 0.9999, 0.9999, 0.9999,
    50.0, 110.0, 0.9999, 2.0
], dtype=float)

initial_parameters_M = np.array([
    0.06008, 0.31087, 0.34431, 0.00698,
    1.98569, 26.7107, 0.00022, 1.08800
], dtype=float)

TOL_PARAMETER = 1e-12
MIN_ALPHA = 2.0 ** -40

# True = residual logaritmik
# False = residual probabilitas asli
USE_LOG_RESIDUAL = True


# ============================================================
# 2. PERSIAPAN DATA
# ============================================================

fit_data_M = (
    df_M[["age", "qx"]]
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_values("age")
    .reset_index(drop=True)
)

fit_data_M = fit_data_M[
    (fit_data_M["age"] >= 0) &
    (fit_data_M["qx"] >= 0) &
    (fit_data_M["qx"] <= 1)
].copy()

x_fit_M = fit_data_M["age"].to_numpy()
y_fit_M = fit_data_M["qx"].to_numpy()

if len(fit_data_M) <= 8:
    raise ValueError("Jumlah observasi harus lebih dari 8.")

if USE_LOG_RESIDUAL and np.any(y_fit_M <= 0):
    raise ValueError(
        "Residual log memerlukan qx aktual > 0. "
        "Periksa observasi qx = 0 sebelum estimasi."
    )

if not np.all(
    (initial_parameters_M > parameter_lower_M) &
    (initial_parameters_M < parameter_upper_M)
):
    raise ValueError("Parameter awal harus berada di dalam batas.")

print("Jumlah observasi:", len(x_fit_M))
print("Rentang usia:", x_fit_M.min(), "-", x_fit_M.max())


# ============================================================
# 3. FUNGSI HELIGMAN-POLLARD
# ============================================================

def hp_components_M(theta, x):

    A, B, C, D, E, F, G, H = theta
    x = np.asarray(x, dtype=float)

    z = x + B
    u = np.log(np.maximum(x, 1e-12) / F)

    log_t1 = np.log(A) * z**C
    log_t2 = np.log(D) - E * u**2
    log_t3_odds = np.log(G) + x * np.log(H)

    t1 = np.exp(np.clip(log_t1, -745, 700))
    t2 = np.exp(np.clip(log_t2, -745, 700))
    t3 = expit(log_t3_odds)

    return t1, t2, t3


def heligman_pollard_qx_M(x, *theta):

    if len(theta) == 1:
        theta = np.asarray(theta[0], dtype=float)
    else:
        theta = np.asarray(theta, dtype=float)

    t1, t2, t3 = hp_components_M(theta, x)

    return t1 + t2 + t3


# ============================================================
# 4. JACOBIAN ANALITIK
# ============================================================

def hp_jacobian_M(theta, x):

    A, B, C, D, E, F, G, H = theta
    x = np.asarray(x, dtype=float)

    t1, t2, t3 = hp_components_M(theta, x)

    z = x + B
    u = np.log(np.maximum(x, 1e-12) / F)

    J = np.zeros((len(x), 8))

    # Komponen mortalitas anak
    J[:, 0] = t1 * z**C / A
    J[:, 1] = t1 * np.log(A) * C * z**(C - 1)
    J[:, 2] = t1 * np.log(A) * z**C * np.log(z)

    # Komponen accident hump
    J[:, 3] = t2 / D
    J[:, 4] = -t2 * u**2
    J[:, 5] = t2 * (2 * E * u / F)

    # Komponen mortalitas lanjut
    J[:, 6] = t3 * (1 - t3) / G
    J[:, 7] = t3 * (1 - t3) * x / H

    return J


# ============================================================
# 5. RESIDUAL DAN JACOBIAN RESIDUAL
# ============================================================

def residual_M(theta):

    q_hat_M = heligman_pollard_qx_M(x_fit_M, theta)

    if not np.all(np.isfinite(q_hat_M)) or np.any(q_hat_M <= 0):
        return np.full(len(x_fit_M), np.nan)

    if USE_LOG_RESIDUAL:
        return np.log(q_hat_M) - np.log(y_fit_M)

    return q_hat_M - y_fit_M


def residual_jacobian_M(theta):

    q_hat_M = heligman_pollard_qx_M(x_fit_M, theta)

    J = hp_jacobian_M(theta, x_fit_M)

    if USE_LOG_RESIDUAL:
        J = J / q_hat_M[:, None]

    return J


def calculate_sse_M(theta):

    r = residual_M(theta)

    if not np.all(np.isfinite(r)):
        return np.inf

    return float(r @ r)


# ============================================================
# 6. TRANSFORMASI SKALA PARAMETER
# ============================================================

# theta = lower + scale * z
# J_z = J_theta * scale

scale_M = parameter_upper_M - parameter_lower_M


def theta_to_z_M(theta):

    return (theta - parameter_lower_M) / scale_M


def z_to_theta_M(z):

    return parameter_lower_M + scale_M * z


# ============================================================
# 7. ITERASI GAUSS-NEWTON
# ============================================================

theta_M = initial_parameters_M.copy()
z_M = theta_to_z_M(theta_M)

history_M = []
success_M = False
reason_M = "Belum konvergen"

iteration_M = 0

while True:

    iteration_M += 1

    theta_M = z_to_theta_M(z_M)

    r_M = residual_M(theta_M)
    J_theta_M = residual_jacobian_M(theta_M)

    if not (
        np.all(np.isfinite(r_M)) and
        np.all(np.isfinite(J_theta_M))
    ):
        reason_M = "Residual atau Jacobian tidak finite"
        break

    sse_old_M = float(r_M @ r_M)

    # Jacobian parameter ternormalisasi
    J_z_M = J_theta_M * scale_M[None, :]

    rank_M = np.linalg.matrix_rank(J_z_M)

    # Penyelesaian Gauss-Newton
    delta_M, _, _, _ = np.linalg.lstsq(
        J_z_M, -r_M, rcond=1e-12
    )

    if not np.all(np.isfinite(delta_M)):
        reason_M = "Arah Gauss-Newton tidak finite"
        break

    gradient_M = J_z_M.T @ r_M

    # ========================================================
    # 8. BACKTRACKING DAN BATAS PARAMETER
    # ========================================================

    alpha_M = 1.0
    accepted_M = False

    while alpha_M >= MIN_ALPHA:

        z_candidate_M = z_M + alpha_M * delta_M

        # Kandidat harus berada dalam batas parameter
        if (
            np.any(z_candidate_M <= 0) or
            np.any(z_candidate_M >= 1)
        ):
            alpha_M *= 0.5
            continue

        theta_candidate_M = z_to_theta_M(z_candidate_M)

        sse_new_M = calculate_sse_M(theta_candidate_M)

        if np.isfinite(sse_new_M) and sse_new_M < sse_old_M:
            accepted_M = True
            break

        alpha_M *= 0.5

    if not accepted_M:
        reason_M = "Tidak ditemukan langkah yang menurunkan SSE"
        break

    # ========================================================
    # 9. KRITERIA KONVERGENSI CAUCHY
    # ========================================================

    parameter_change_M = np.linalg.norm(
        z_candidate_M - z_M, ord=np.inf
    )

    relative_sse_change_M = (
        abs(sse_old_M - sse_new_M) /
        max(1.0, abs(sse_old_M))
    )

    history_M.append({
        "Iterasi": iteration_M,
        "SSE": sse_new_M,
        "Alpha": alpha_M,
        "Perubahan_Parameter": parameter_change_M,
        "Perubahan_SSE": relative_sse_change_M,
        "Rank_Jacobian": rank_M,
        **dict(zip(parameter_names_M, theta_candidate_M))
    })

    z_M = z_candidate_M

    if iteration_M == 1 or iteration_M % 100 == 0:
        print(
            f"Iterasi {iteration_M:5d} | "
            f"SSE = {sse_new_M:.12e} | "
            f"Delta = {parameter_change_M:.3e} | "
            f"Alpha = {alpha_M:.3e}"
        )

    # Kriteria Cauchy
    if parameter_change_M < TOL_PARAMETER:

        success_M = True
        reason_M = "Konvergen: memenuhi kriteria Cauchy"

        print(
            f"\nKonvergen pada iterasi ke-{iteration_M}"
        )

        break


# ============================================================
# 10. HASIL ESTIMASI
# ============================================================

theta_hat_M = z_to_theta_M(z_M)

q_hat_M = heligman_pollard_qx_M(
    x_fit_M, theta_hat_M
)

r_final_M = residual_M(theta_hat_M)

J_final_M = residual_jacobian_M(theta_hat_M)

SSE_objective_M = float(r_final_M @ r_final_M)

SSE_original_M = float(
    np.sum((q_hat_M - y_fit_M)**2)
)

RMSE_original_M = np.sqrt(
    np.mean((q_hat_M - y_fit_M)**2)
)

if USE_LOG_RESIDUAL:
    RMSE_log_M = np.sqrt(np.mean(r_final_M**2))
else:
    RMSE_log_M = np.nan

gradient_final_M = J_final_M.T @ r_final_M

J_scaled_final_M = J_final_M * scale_M[None, :]

rank_final_M = np.linalg.matrix_rank(J_scaled_final_M)

condition_number_M = np.linalg.cond(J_scaled_final_M)

df_parameters_M = pd.DataFrame({
    "Parameter": parameter_names_M,
    "Batas Bawah": parameter_lower_M,
    "Nilai Awal": initial_parameters_M,
    "Estimasi": theta_hat_M,
    "Batas Atas": parameter_upper_M
})

df_history_M = pd.DataFrame(history_M)

print("\n" + "=" * 60)
print("HASIL ESTIMASI HELIGMAN-POLLARD MALE")
print("=" * 60)

print("Success             :", success_M)
print("Status              :", reason_M)
print("Jumlah iterasi      :", len(history_M))
print("Tolerance Cauchy    :", TOL_PARAMETER)
print("SSE objektif        :", SSE_objective_M)
print("SSE skala asli      :", SSE_original_M)
print("RMSE skala asli     :", RMSE_original_M)
print("RMSE log            :", RMSE_log_M)
print("Rank Jacobian       :", rank_final_M, "/ 8")
print("Condition Number    :", condition_number_M)

print(
    "Norm gradien scaled :",
    np.linalg.norm(
        J_scaled_final_M.T @ r_final_M,
        ord=np.inf
    )
)

print("\nEstimasi parameter:")

print(
    df_parameters_M.to_string(
        index=False,
        float_format=lambda v: f"{v:.10g}"
    )
)


# ============================================================
# 11. MATRIKS KOVARIANS DAN STANDARD ERROR
# ============================================================

n_M = len(x_fit_M)
p_M = 8

df_residual_M = n_M - p_M

if df_residual_M > 0 and rank_final_M == p_M:

    sigma2_M = SSE_objective_M / df_residual_M

    covariance_M = sigma2_M * np.linalg.pinv(
        J_final_M.T @ J_final_M
    )

    standard_error_M = np.sqrt(
        np.maximum(np.diag(covariance_M), 0)
    )

else:

    sigma2_M = np.nan

    covariance_M = np.full(
        (p_M, p_M), np.nan
    )

    standard_error_M = np.full(p_M, np.nan)


df_parameters_M["Standard Error"] = standard_error_M

df_covariance_M = pd.DataFrame(
    covariance_M,
    index=parameter_names_M,
    columns=parameter_names_M
)

df_jacobian_M = pd.DataFrame(
    J_final_M,
    columns=parameter_names_M
)

df_jacobian_M.insert(0, "age", x_fit_M)

print("\nVarians residual:", sigma2_M)

print("\nParameter dan Standard Error:")

print(
    df_parameters_M[
        ["Parameter", "Estimasi", "Standard Error"]
    ].to_string(index=False)
)

print("\nMatriks kovarians:")

print(df_covariance_M)


# ============================================================
# 12. DATA HASIL FITTING
# ============================================================

df_fitted_M = pd.DataFrame({
    "age": x_fit_M,
    "qx_actual": y_fit_M,
    "qx_fitted": q_hat_M,
    "residual": q_hat_M - y_fit_M,
    "residual_log": (
        np.log(q_hat_M) - np.log(y_fit_M)
        if np.all(y_fit_M > 0)
        else np.nan
    )
})


# ============================================================
# 13. GRAFIK HASIL ESTIMASI MALE
# ============================================================

import matplotlib as mpl
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

# ============================================================
# A. PENGATURAN FONT DAN STYLE
# ============================================================

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Georgia', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'font.size': 22,
    'axes.titlesize': 26,
    'axes.labelsize': 24,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 19,
    'axes.grid': False
})

# ============================================================
# B. GRAFIK SKALA ASLI
# ============================================================

fig, ax = plt.subplots(figsize=(13, 6.5))

ax.plot(
    x_fit_M,
    y_fit_M,
    color='black',
    linestyle='--',
    linewidth=2.5,
    label='GAM 71'
)

ax.plot(
    x_fit_M,
    q_hat_M,
    color='green',
    linestyle='-',
    linewidth=2.5,
    label='Estimasi Heligman-Pollard'
)

ax.set_title(
    r'Hasil Estimasi Mortalitas $\mathit{Male}$',
    fontsize=28,
    pad=18
)

ax.set_xlabel(
    'Usia (tahun)',
    fontsize=24,
    labelpad=12
)

ax.set_ylabel(
    r'Probabilitas Kematian (${}^{M}q_x$)',
    fontsize=24,
    labelpad=12
)

ax.set_xlim(0, 110)
ax.tick_params(axis='both', labelsize=20)
ax.grid(False)

ax.legend(
    loc='best',
    frameon=True,
    fontsize=19
)

plt.tight_layout()

plt.savefig(
    'Estimasi_HP_Male_Skala_Asli.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ============================================================
# C. GRAFIK LOGARITMIK
# ============================================================

if np.all(y_fit_M > 0) and np.all(q_hat_M > 0):

    fig, ax = plt.subplots(figsize=(13, 6.5))

    # Kurva aktual GAM 71
    ax.semilogy(
        x_fit_M,
        y_fit_M,
        color='black',
        linestyle='--',
        linewidth=2.5
    )

    # Segmen kurva estimasi Heligman-Pollard
    points_M = np.column_stack((
        x_fit_M,
        q_hat_M
    ))

    segments_M = np.stack(
        [points_M[:-1], points_M[1:]],
        axis=1
    )

    # Posisi estimasi terhadap GAM 71
    log_difference_M = (
        np.log(q_hat_M) - np.log(y_fit_M)
    )

    segment_difference_M = (
        log_difference_M[:-1] +
        log_difference_M[1:]
    ) / 2

    colors_M = np.where(
        segment_difference_M >= 0,
        'green',
        'red'
    )

    # Garis HP berwarna hijau atau merah
    hp_line_M = LineCollection(
        segments_M,
        colors=colors_M,
        linewidths=2.8,
        linestyles='solid'
    )

    ax.add_collection(hp_line_M)

    ax.set_yscale('log')

    ax.set_title(
        r'Hasil Estimasi Mortalitas $\mathit{Male}$',
        fontsize=28,
        pad=18
    )

    ax.set_xlabel(
        'Usia (tahun)',
        fontsize=24,
        labelpad=12
    )

    ax.set_ylabel(
        r'Probabilitas Kematian (${}^{M}q_x$)',
        fontsize=24,
        labelpad=12
    )

    ax.set_xlim(0, 110)

    ax.set_ylim(
        min(y_fit_M.min(), q_hat_M.min()) * 0.8,
        max(y_fit_M.max(), q_hat_M.max()) * 1.2
    )

    ax.tick_params(axis='both', labelsize=20)
    ax.grid(False)

    # Legenda
    legend_elements_M = [
        Line2D(
            [0], [0],
            color='black',
            linestyle='--',
            linewidth=2.5,
            label='GAM 71'
        ),
        Line2D(
            [0], [0],
            color='green',
            linestyle='-',
            linewidth=2.8,
            label='Estimasi di atas GAM 71'
        ),
        Line2D(
            [0], [0],
            color='red',
            linestyle='-',
            linewidth=2.8,
            label='Estimasi di bawah GAM 71'
        )
    ]

    ax.legend(
        handles=legend_elements_M,
        loc='best',
        frameon=True,
        fontsize=19
    )

    plt.tight_layout()

    plt.savefig(
        'Estimasi_HP_Male.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.savefig(
        'Estimasi_HP_Male.pdf',
        bbox_inches='tight'
    )

    plt.show()

# ============================================================
# D. GRAFIK KONVERGENSI GAUSS-NEWTON
# ============================================================

if not df_history_M.empty:

    fig, ax = plt.subplots(figsize=(13, 6.5))

    ax.plot(
        df_history_M['Iterasi'],
        df_history_M['SSE'],
        color='blue',
        linewidth=2.8,
        label='SSE'
    )

    ax.set_title(
        r'Konvergensi Gauss-Newton $\mathit{Male}$',
        fontsize=28,
        pad=18
    )

    ax.set_xlabel(
        'Iterasi',
        fontsize=24,
        labelpad=12
    )

    ax.set_ylabel(
        'Jumlah Kuadrat Residual (SSE)',
        fontsize=24,
        labelpad=12
    )

    ax.tick_params(axis='both', labelsize=20)
    ax.grid(False)

    ax.legend(
        loc='best',
        frameon=True,
        fontsize=19
    )

    plt.tight_layout()

    plt.savefig(
        'Konvergensi_HP_Male.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.savefig(
        'Konvergensi_HP_Male.pdf',
        bbox_inches='tight'
    )

    plt.show()

# ============================================================
# 14. EKSPOR HASIL KE EXCEL
# ============================================================

output_file_M = (
    "Estimasi_Heligman_Pollard_Gauss_Newton_Male_Sharrow.xlsx"
)

with pd.ExcelWriter(output_file_M) as writer:

    df_parameters_M.to_excel(
        writer,
        sheet_name="Parameter",
        index=False
    )

    df_fitted_M.to_excel(
        writer,
        sheet_name="Fitted",
        index=False
    )

    df_history_M.to_excel(
        writer,
        sheet_name="Iterasi",
        index=False
    )

    df_jacobian_M.to_excel(
        writer,
        sheet_name="Jacobian",
        index=False
    )

    df_covariance_M.to_excel(
        writer,
        sheet_name="Kovarians"
    )

print("\nFile Excel tersimpan:", output_file_M)


# Pemeriksaan Konvergensi Monte Carlo — Male


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. KONFIGURASI MONTE CARLO MALE
# ============================================================

TOL = 1e-12
MIN_SIM = 10000
CONSECUTIVE = 5
BATCH_SIZE = 2000
SEED = 71

ages = np.arange(0, 111, dtype=float)
parameter_names_M = parameter_names_M.copy()

# ============================================================
# 2. HASIL ESTIMASI MALE
# ============================================================

theta_mean_M = np.asarray(
    theta_hat_M, dtype=float
).copy()

covariance_M = (
    df_covariance_M.loc[
        parameter_names_M,
        parameter_names_M
    ]
    .to_numpy(dtype=float)
    .copy()
)

# Menggunakan batas parameter dari estimasi Male
lower_M = np.asarray(
    parameter_lower_M, dtype=float
).copy()

upper_M = np.asarray(
    parameter_upper_M, dtype=float
).copy()

# Validasi parameter dan kovarians
if theta_mean_M.shape != (8,):
    raise ValueError("theta_hat_M harus memiliki 8 parameter.")

if covariance_M.shape != (8, 8):
    raise ValueError("Kovarians Male harus berukuran 8x8.")

if not np.all(np.isfinite(theta_mean_M)):
    raise ValueError("Parameter Male mengandung NaN/inf.")

if not np.all(np.isfinite(covariance_M)):
    raise ValueError("Kovarians Male mengandung NaN/inf.")

if not np.all(
    (theta_mean_M >= lower_M) &
    (theta_mean_M <= upper_M)
):
    raise ValueError("Parameter Male berada di luar bounds.")

print("HASIL ESTIMASI MALE")

display(pd.DataFrame({
    "Parameter": parameter_names_M,
    "Theta_Hat": theta_mean_M,
    "Batas_Bawah": lower_M,
    "Batas_Atas": upper_M
}))


# ============================================================
# 3. CHOLESKY DECOMPOSITION
# ============================================================

# Standardisasi matriks kovarians
std_M = np.sqrt(np.diag(covariance_M))

if not np.all(np.isfinite(std_M)) or np.any(std_M <= 0):
    raise ValueError("Varians parameter tidak valid.")

corr_M = covariance_M / np.outer(std_M, std_M)

# Cholesky pada matriks korelasi
try:
    L_corr_M = np.linalg.cholesky(corr_M)

except np.linalg.LinAlgError:
    print("Eigenvalue korelasi:")
    print(np.linalg.eigvalsh(corr_M))

    raise ValueError(
        "Matriks kovarians tidak positive definite."
    )

# Mengembalikan ke skala kovarians asli
L_M = std_M[:, None] * L_corr_M

# Verifikasi Cholesky
cov_rekonstruksi_M = L_M @ L_M.T

error_rekonstruksi_M = (
    np.linalg.norm(
        cov_rekonstruksi_M - covariance_M
    )
    / np.linalg.norm(covariance_M)
)

print("\nHASIL CHOLESKY DECOMPOSITION")
print("Error rekonstruksi:", error_rekonstruksi_M)

if error_rekonstruksi_M > 1e-8:
    raise ValueError("Rekonstruksi kovarians tidak akurat.")

rng_M = np.random.default_rng(SEED)


# ============================================================
# 4. FUNGSI HELIGMAN-POLLARD
# ============================================================

def hp_qx_batch_M(theta_batch, ages):

    A, B, C, D, E, F, G, H = (
        theta_batch[:, i:i+1]
        for i in range(8)
    )

    x = ages.reshape(1, -1)
    x_safe = np.maximum(x, 1e-12)

    # Komponen mortalitas anak
    t1 = np.exp(np.log(A) * (x + B)**C)

    # Komponen accident hump
    t2 = np.exp(np.log(D) - E * (np.log(x_safe / F))**2)

    # Komponen mortalitas lanjut
    log_odds = np.log(G) + x * np.log(H)
    t3 = 1 / (1 + np.exp(-log_odds))

    return t1 + t2 + t3


# ============================================================
# 5. INISIALISASI SIMULASI
# ============================================================

n_M = 0
jumlah_dibangkitkan_M = 0

# Rata-rata kumulatif mortalitas
Qx_mean_M = np.zeros(len(ages), dtype=float)

# Perubahan rata-rata kumulatif
delta_Qx_M = np.full(len(ages), np.nan)

# Penghitung pemenuhan toleransi berturut-turut
streak_M = np.zeros(len(ages), dtype=int)

# Iterasi pertama setiap usia memenuhi toleransi
n_konvergen_M = np.full(
    len(ages), -1, dtype=int
)

# Nilai rata-rata saat pertama kali memenuhi toleransi
Qx_konvergen_M = np.full(
    len(ages), np.nan
)

# ============================================================
# 6. ITERASI MONTE CARLO DENGAN KRITERIA CAUCHY
# ============================================================

while True:

    ukuran_batch_M = BATCH_SIZE

    # --------------------------------------------------------
    # A. PEMBANGKITAN PARAMETER NORMAL MULTIVARIAT
    # --------------------------------------------------------

    Z_M = rng_M.standard_normal(
        (ukuran_batch_M, 8)
    )

    theta_batch_M = (
        theta_mean_M +
        Z_M @ L_M.T
    )

    jumlah_dibangkitkan_M += ukuran_batch_M

    # --------------------------------------------------------
    # B. VALIDASI BATAS PARAMETER
    # --------------------------------------------------------

    valid_theta_M = np.all(
        (theta_batch_M >= lower_M) &
        (theta_batch_M <= upper_M),
        axis=1
    )

    theta_batch_M = theta_batch_M[valid_theta_M]

    if len(theta_batch_M) == 0:
        continue

    # --------------------------------------------------------
    # C. HITUNG qx SELURUH USIA
    # --------------------------------------------------------

    qx_batch_M = hp_qx_batch_M(
        theta_batch_M, ages
    )

    # Validasi probabilitas mortalitas
    valid_qx_M = (
        np.all(np.isfinite(qx_batch_M), axis=1) &
        np.all(
            (qx_batch_M >= 0) &
            (qx_batch_M <= 1),
            axis=1
        )
    )

    qx_batch_M = qx_batch_M[valid_qx_M]

    # --------------------------------------------------------
    # D. PERBARUI RATA-RATA KUMULATIF
    # --------------------------------------------------------

    for qx_j_M in qx_batch_M:

        n_M += 1

        Qx_sebelumnya_M = Qx_mean_M.copy()

        # Rata-rata kumulatif Monte Carlo
        Qx_mean_M += (
            qx_j_M - Qx_mean_M
        ) / n_M

        if n_M < 2:
            continue

        # ----------------------------------------------------
        # E. KRITERIA CAUCHY
        # ----------------------------------------------------

        # Selisih dua rata-rata kumulatif berurutan
        delta_Qx_M = np.abs(
            Qx_mean_M - Qx_sebelumnya_M
        )

        # Konvergensi hanya diperiksa setelah minimum simulasi
        if n_M >= MIN_SIM:

            belum_konvergen_M = (
                n_konvergen_M == -1
            )

            memenuhi_M = (
                delta_Qx_M < TOL
            )

            # Menambah streak jika memenuhi toleransi
            streak_M[
                belum_konvergen_M & memenuhi_M
            ] += 1

            # Mengulang streak jika tidak memenuhi toleransi
            streak_M[
                belum_konvergen_M & ~memenuhi_M
            ] = 0

            # Usia yang baru memenuhi kriteria
            baru_konvergen_M = (
                belum_konvergen_M &
                (streak_M >= CONSECUTIVE)
            )

            # Mencatat iterasi konvergensi pertama
            n_konvergen_M[
                baru_konvergen_M
            ] = n_M

            # Menyimpan nilai rata-rata saat konvergen
            Qx_konvergen_M[
                baru_konvergen_M
            ] = Qx_mean_M[
                baru_konvergen_M
            ]

        # ----------------------------------------------------
        # F. PENGECEKAN SELURUH USIA
        # ----------------------------------------------------

        if np.all(n_konvergen_M != -1):
            break

    # --------------------------------------------------------
    # G. INFORMASI PROGRES SIMULASI
    # --------------------------------------------------------

    if n_M // 10000 > (n_M - len(qx_batch_M)) // 10000:

        jumlah_selesai_M = np.sum(
            n_konvergen_M != -1
        )

        print(
            f"Iterasi {n_M:,} | "
            f"Usia memenuhi kriteria: "
            f"{jumlah_selesai_M}/111"
        )

    # Berhenti jika seluruh usia memenuhi kriteria
    if np.all(n_konvergen_M != -1):
        break


# ============================================================
# 7. JUMLAH ITERASI KUMULATIF PER USIA
# ============================================================

n_digunakan_M = np.full(
    len(ages),
    np.nan
)

maksimum_sebelumnya_M = 0

for i, usia in enumerate(ages):

    if n_konvergen_M[i] != -1:

        maksimum_sebelumnya_M = max(
            maksimum_sebelumnya_M,
            int(n_konvergen_M[i])
        )

        n_digunakan_M[i] = maksimum_sebelumnya_M


# ============================================================
# 8. TABEL HASIL KONVERGENSI MALE
# ============================================================

df_konvergensi_M = pd.DataFrame({

    "Usia": ages.astype(int),

    "Iterasi_Konvergen": np.where(
        n_konvergen_M == -1,
        np.nan,
        n_konvergen_M.astype(float)
    ),

    "Iterasi_Digunakan": n_digunakan_M,

    "Qx_Saat_Konvergen": Qx_konvergen_M,

    "Qx_Akhir": Qx_mean_M,

    "Perubahan_Qx_Akhir": delta_Qx_M
})


print("\n" + "=" * 60)
print("HASIL MONTE CARLO - MALE")
print("=" * 60)

print("Toleransi              :", TOL)
print("Minimum simulasi       :", MIN_SIM)
print("Konvergensi berturut   :", CONSECUTIVE)
print("Total simulasi valid   :", n_M)
print("Total dibangkitkan     :", jumlah_dibangkitkan_M)

print(
    "Jumlah usia konvergen  :",
    np.sum(n_konvergen_M != -1),
    "/ 111"
)

print(
    "Acceptance rate        :",
    f"{100*n_M/jumlah_dibangkitkan_M:.4f}%"
)

print(
    "Jumlah iterasi akhir   :",
    int(np.max(n_digunakan_M))
)

print("Status: Seluruh usia memenuhi kriteria Cauchy.")

display(df_konvergensi_M)


# ============================================================
# 9. GRAFIK JUMLAH ITERASI
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(
    df_konvergensi_M["Usia"],
    df_konvergensi_M["Iterasi_Konvergen"],
    label="Iterasi pertama memenuhi toleransi",
    alpha=0.7
)

plt.plot(
    df_konvergensi_M["Usia"],
    df_konvergensi_M["Iterasi_Digunakan"],
    label="Iterasi kumulatif maksimum",
    linewidth=2
)

plt.xlabel("Usia (x)")
plt.ylabel("Jumlah Iterasi")
plt.title("Konvergensi Monte Carlo Male")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 10. GRAFIK PERUBAHAN RATA-RATA
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(
    ages,
    delta_Qx_M,
    linewidth=1.5
)

plt.axhline(
    TOL,
    color="red",
    linestyle="--",
    label="Tolerance"
)

plt.yscale("log")

plt.xlabel("Usia (x)")
plt.ylabel("Perubahan Rata-rata Qx")
plt.title("Perubahan Rata-rata Kumulatif Monte Carlo Male")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()


# ============================================================
# 11. SIMPAN HASIL
# ============================================================

output_file_M = "Konvergensi_Monte_Carlo_Male.xlsx"

with pd.ExcelWriter(output_file_M) as writer:

    df_konvergensi_M.to_excel(
        writer,
        sheet_name="Konvergensi",
        index=False
    )

    pd.DataFrame({
        "Usia": ages.astype(int),
        "Qx_Mean": Qx_mean_M,
        "Delta_Qx": delta_Qx_M
    }).to_excel(
        writer,
        sheet_name="Rata_Rata_Qx",
        index=False
    )

print("\nFile tersimpan:")
print(output_file_M)


# Pembentukan Selang Mortalitas — Male


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from numba import njit

# ============================================================
# 1. KONFIGURASI SIMULASI MONTE CARLO MALE
# ============================================================

# Jumlah simulasi dari hasil konvergensi sebelumnya
N_SIM_M = int(
    df_konvergensi_M["Iterasi_Digunakan"].iloc[-1]
) + 1000000

SEED_M = 71
BATCH_SIZE_M = 2000

ages_M = np.arange(0, 111, dtype=float)
parameter_names_M = list(parameter_names_M)

# Histogram untuk menghitung persentil tanpa
# menyimpan seluruh hasil simulasi di RAM/disk
N_BINS_M = 100000
LOG_MIN_M = -20.0
LOG_MAX_M = 0.0

if N_SIM_M <= 0:
    raise ValueError("Jumlah simulasi harus positif.")

if N_SIM_M > np.iinfo(np.uint32).max:
    raise ValueError(
        "Jumlah simulasi melebihi kapasitas histogram uint32."
    )

print("=" * 60)
print("KONFIGURASI MONTE CARLO MALE")
print("=" * 60)

print("Jumlah simulasi valid :", f"{N_SIM_M:,}")
print("Jumlah usia           :", len(ages_M))
print("Batch size            :", BATCH_SIZE_M)
print("Jumlah bin histogram  :", N_BINS_M)


# ============================================================
# 2. HASIL ESTIMASI HELIGMAN-POLLARD MALE
# ============================================================

theta_mean_M = np.asarray(
    theta_hat_M, dtype=float
).copy()

covariance_M = (
    df_covariance_M.loc[
        parameter_names_M,
        parameter_names_M
    ]
    .to_numpy(dtype=float)
    .copy()
)

# Batas parameter sesuai estimasi Male
lower_M = np.asarray(
    parameter_lower_M, dtype=float
).copy()

upper_M = np.asarray(
    parameter_upper_M, dtype=float
).copy()

# Validasi
if theta_mean_M.shape != (8,):
    raise ValueError("theta_hat_M harus memiliki 8 parameter.")

if covariance_M.shape != (8, 8):
    raise ValueError("Kovarians harus berukuran 8x8.")

if not np.all(np.isfinite(theta_mean_M)):
    raise ValueError("Parameter Male mengandung NaN/inf.")

if not np.all(np.isfinite(covariance_M)):
    raise ValueError("Kovarians Male mengandung NaN/inf.")

if not np.all(
    (theta_mean_M >= lower_M) &
    (theta_mean_M <= upper_M)
):
    raise ValueError("Parameter Male berada di luar batas.")

print("\nHASIL ESTIMASI MALE")

display(pd.DataFrame({
    "Parameter": parameter_names_M,
    "Theta_Hat": theta_mean_M,
    "Batas_Bawah": lower_M,
    "Batas_Atas": upper_M
}))


# ============================================================
# 3. CHOLESKY DECOMPOSITION
# ============================================================

std_M = np.sqrt(np.diag(covariance_M))

if not np.all(np.isfinite(std_M)) or np.any(std_M <= 0):
    raise ValueError("Varians parameter tidak valid.")

corr_M = covariance_M / np.outer(std_M, std_M)

try:
    L_corr_M = np.linalg.cholesky(corr_M)

except np.linalg.LinAlgError:

    print("Eigenvalue korelasi:")
    print(np.linalg.eigvalsh(corr_M))

    raise ValueError(
        "Matriks korelasi tidak positive definite."
    )

L_M = std_M[:, None] * L_corr_M

cov_rekonstruksi_M = L_M @ L_M.T

error_rekonstruksi_M = (
    np.linalg.norm(
        cov_rekonstruksi_M - covariance_M
    )
    / np.linalg.norm(covariance_M)
)

print("\nHASIL CHOLESKY DECOMPOSITION")
print("Error rekonstruksi:", error_rekonstruksi_M)

if error_rekonstruksi_M > 1e-8:
    raise ValueError(
        "Rekonstruksi matriks kovarians tidak akurat."
    )

rng_M = np.random.default_rng(SEED_M)


# ============================================================
# 4. FUNGSI HELIGMAN-POLLARD
# ============================================================

def hp_qx_simulasi_M(theta_batch_M, ages_M):

    A, B, C, D, E, F, G, H = (
        theta_batch_M[:, i:i+1]
        for i in range(8)
    )

    x_M = ages_M.reshape(1, -1)

    z_M = x_M + B

    u_M = np.log(
        np.maximum(x_M, 1e-12) / F
    )

    # Komponen mortalitas anak
    t1_M = np.exp(np.clip(
        np.log(A) * z_M**C,
        -745, 700
    ))

    # Komponen accident hump
    t2_M = np.exp(np.clip(
        np.log(D) - E * u_M**2,
        -745, 700
    ))

    # Komponen mortalitas usia lanjut
    log_odds_M = np.log(G) + x_M * np.log(H)

    t3_M = expit(log_odds_M)

    return t1_M + t2_M + t3_M


# ============================================================
# 5. HISTOGRAM MONTE CARLO DI RAM
# ============================================================

# Histogram hanya menyimpan frekuensi setiap bin,
# bukan seluruh sampel qx.

hist_M = np.zeros(
    (len(ages_M), N_BINS_M),
    dtype=np.uint32
)

# Lebar bin pada skala log10
lebar_bin_M = (
    LOG_MAX_M - LOG_MIN_M
) / N_BINS_M

# Rata-rata mortalitas dikumpulkan secara kumulatif
jumlah_qx_M = np.zeros(
    len(ages_M), dtype=np.float64
)

jumlah_theta_M = np.zeros(
    8, dtype=np.float64
)

jumlah_valid_M = 0
jumlah_dibangkitkan_M = 0
jumlah_batch_M = 0

# Menghitung nilai qx yang berada di bawah
# batas minimum histogram
jumlah_bawah_hist_M = 0

print("\nPERSIAPAN PENYIMPANAN")

print(
    "Memori histogram:",
    f"{hist_M.nbytes / 1024**2:.2f} MiB"
)

print("Tidak menyimpan matriks seluruh simulasi.")


# ============================================================
# 6. FUNGSI PEMBARUAN HISTOGRAM
# ============================================================

# Numba mempercepat pembaruan frekuensi histogram.
# Fungsi ini tidak menyimpan qx dari seluruh simulasi.

@njit
def update_histogram_M(hist, indeks_bin):

    n_batch, n_usia = indeks_bin.shape

    for i in range(n_batch):
        for j in range(n_usia):

            k = indeks_bin[i, j]

            hist[j, k] += 1


# ============================================================
# 7. PEMBANGKITAN PARAMETER MONTE CARLO
# ============================================================

print("\nMEMULAI SIMULASI MONTE CARLO MALE")

while jumlah_valid_M < N_SIM_M:

    # --------------------------------------------------------
    # A. PEMBANGKITAN NORMAL MULTIVARIAT
    # --------------------------------------------------------

    Z_M = rng_M.standard_normal(
        (BATCH_SIZE_M, 8)
    )

    theta_batch_M = (
        theta_mean_M +
        Z_M @ L_M.T
    )

    jumlah_dibangkitkan_M += BATCH_SIZE_M

    # --------------------------------------------------------
    # B. VALIDASI BATAS PARAMETER
    # --------------------------------------------------------

    valid_theta_M = np.all(
        (theta_batch_M >= lower_M) &
        (theta_batch_M <= upper_M),
        axis=1
    )

    theta_batch_M = theta_batch_M[valid_theta_M]

    if len(theta_batch_M) == 0:
        continue

    # --------------------------------------------------------
    # C. HITUNG qx SELURUH USIA
    # --------------------------------------------------------

    qx_batch_M = hp_qx_simulasi_M(
        theta_batch_M,
        ages_M
    )

    # --------------------------------------------------------
    # D. VALIDASI PROBABILITAS MORTALITAS
    # --------------------------------------------------------

    valid_qx_M = (
        np.all(np.isfinite(qx_batch_M), axis=1) &
        np.all(
            (qx_batch_M >= 0) &
            (qx_batch_M <= 1),
            axis=1
        )
    )

    theta_batch_M = theta_batch_M[valid_qx_M]
    qx_batch_M = qx_batch_M[valid_qx_M]

    jumlah_ambil_M = min(
        len(qx_batch_M),
        N_SIM_M - jumlah_valid_M
    )

    if jumlah_ambil_M == 0:
        continue

    theta_batch_M = theta_batch_M[:jumlah_ambil_M]
    qx_batch_M = qx_batch_M[:jumlah_ambil_M]

    # --------------------------------------------------------
    # E. AKUMULASI RATA-RATA
    # --------------------------------------------------------

    jumlah_qx_M += np.sum(
        qx_batch_M,
        axis=0
    )

    jumlah_theta_M += np.sum(
        theta_batch_M,
        axis=0
    )

    # --------------------------------------------------------
    # F. KONVERSI qx KE INDEKS HISTOGRAM
    # --------------------------------------------------------

    jumlah_bawah_hist_M += np.count_nonzero(
        qx_batch_M < 10.0**LOG_MIN_M
    )

    log_qx_M = np.log10(
        np.maximum(
            qx_batch_M,
            10.0**LOG_MIN_M
        )
    )

    indeks_bin_M = np.floor(
        (log_qx_M - LOG_MIN_M) /
        lebar_bin_M
    ).astype(np.int32)

    indeks_bin_M = np.clip(
        indeks_bin_M,
        0,
        N_BINS_M - 1
    )

    # --------------------------------------------------------
    # G. PEMBARUAN HISTOGRAM
    # --------------------------------------------------------

    update_histogram_M(
        hist_M,
        indeks_bin_M
    )

    jumlah_valid_M += jumlah_ambil_M
    jumlah_batch_M += 1

    # --------------------------------------------------------
    # H. INFORMASI PROGRES
    # --------------------------------------------------------

    if jumlah_batch_M % 500 == 0:

        print(
            f"Simulasi valid: "
            f"{jumlah_valid_M:,}/{N_SIM_M:,} | "
            f"Dibangkitkan: {jumlah_dibangkitkan_M:,}"
        )


# ============================================================
# 8. VALIDASI HASIL SIMULASI
# ============================================================

print("\n" + "=" * 60)
print("PEMBANGKITAN MONTE CARLO SELESAI")
print("=" * 60)

print(
    "Total simulasi valid:",
    f"{jumlah_valid_M:,}"
)

print(
    "Total dibangkitkan:",
    f"{jumlah_dibangkitkan_M:,}"
)

print(
    "Acceptance rate:",
    f"{100 * jumlah_valid_M / jumlah_dibangkitkan_M:.4f}%"
)

print(
    "Jumlah qx di bawah batas histogram:",
    jumlah_bawah_hist_M
)

# Setiap usia harus mempunyai N_SIM_M data
total_hist_M = np.sum(
    hist_M,
    axis=1,
    dtype=np.int64
)

if not np.all(total_hist_M == N_SIM_M):
    raise ValueError(
        "Frekuensi histogram tidak sesuai jumlah simulasi."
    )

if jumlah_bawah_hist_M > 0:
    print(
        "PERINGATAN: Ada nilai qx di bawah batas "
        "histogram. Akurasi persentil bawah perlu diperiksa."
    )


# ============================================================
# 9. PERHITUNGAN SELANG MORTALITAS 95%
# ============================================================

# Rata-rata Monte Carlo dihitung dari seluruh qx valid
qx_mean_M = jumlah_qx_M / N_SIM_M

qx_lower_M = np.empty(
    len(ages_M), dtype=float
)

qx_upper_M = np.empty(
    len(ages_M), dtype=float
)

# Estimasi kuantil dari histogram logaritmik
# Menggunakan interpolasi dalam bin.

def quantile_histogram_M(hist_usia, p):

    frekuensi_kumulatif = np.cumsum(
        hist_usia,
        dtype=np.int64
    )

    target = p * N_SIM_M

    indeks = np.searchsorted(
        frekuensi_kumulatif,
        target,
        side="left"
    )

    indeks = min(
        int(indeks),
        N_BINS_M - 1
    )

    kumulatif_sebelum = (
        frekuensi_kumulatif[indeks - 1]
        if indeks > 0
        else 0
    )

    frekuensi_bin = int(hist_usia[indeks])

    if frekuensi_bin == 0:
        fraksi = 0.0
    else:
        fraksi = (
            target - kumulatif_sebelum
        ) / frekuensi_bin

    fraksi = np.clip(
        fraksi, 0.0, 1.0
    )

    nilai_log = (
        LOG_MIN_M +
        (indeks + fraksi) * lebar_bin_M
    )

    return 10.0**nilai_log


for i_M in range(len(ages_M)):

    qx_lower_M[i_M] = quantile_histogram_M(
        hist_M[i_M],
        0.025
    )

    qx_upper_M[i_M] = quantile_histogram_M(
        hist_M[i_M],
        0.975
    )

print("\nSELURUH PERSENTIL SELESAI DIHITUNG")

# Histogram tidak diperlukan lagi
del hist_M


# ============================================================
# 10. ESTIMASI TITIK HELIGMAN-POLLARD
# ============================================================

qx_estimasi_M = hp_qx_simulasi_M(
    theta_mean_M.reshape(1, 8),
    ages_M
)[0]

if not np.all(np.isfinite(qx_mean_M)):
    raise ValueError("Mean mortalitas tidak finite.")

if not np.all(
    np.isfinite(qx_lower_M) &
    np.isfinite(qx_upper_M)
):
    raise ValueError("Selang mortalitas tidak finite.")

if not np.all(qx_lower_M <= qx_upper_M):
    raise ValueError("Batas selang mortalitas tidak valid.")


# ============================================================
# 11. TABEL SELANG MORTALITAS MALE
# ============================================================

df_selang_M = pd.DataFrame({
    "Usia": ages_M.astype(int),
    "qx_lower": qx_lower_M,
    "qx_mean": qx_mean_M,
    "qx_upper": qx_upper_M,
    "qx_estimasi": qx_estimasi_M
})

# Data GAM 71 Male
df_GAM_M = df_M[["age", "qx"]].copy()

df_GAM_M.columns = [
    "Usia",
    "qx_GAM71"
]

df_GAM_M["Usia"] = (
    df_GAM_M["Usia"].astype(int)
)

df_selang_M = df_selang_M.merge(
    df_GAM_M,
    on="Usia",
    how="left"
)

# Urutan kolom sesuai kebutuhan
df_selang_M = df_selang_M[
    [
        "Usia",
        "qx_lower",
        "qx_mean",
        "qx_upper",
        "qx_estimasi",
        "qx_GAM71"
    ]
]

print("\nHASIL SELANG MORTALITAS MALE")

display(df_selang_M)


# ============================================================
# 12. VALIDASI RATA-RATA PARAMETER
# ============================================================

mean_theta_simulasi_M = (
    jumlah_theta_M / N_SIM_M
)

df_validasi_M = pd.DataFrame({
    "Parameter": parameter_names_M,
    "Theta_Hat": theta_mean_M,
    "Mean_Simulasi": mean_theta_simulasi_M,
    "Selisih": (
        mean_theta_simulasi_M - theta_mean_M
    )
})

print("\nVALIDASI RATA-RATA PARAMETER MALE")

display(df_validasi_M)

print(
    "\nSelisih terbesar qx_mean dan qx_estimasi:",
    np.max(
        np.abs(qx_mean_M - qx_estimasi_M)
    )
)


# ============================================================
# 13. GRAFIK SELANG MORTALITAS SKALA ASLI
# ============================================================

plt.figure(figsize=(12, 5))

plt.fill_between(
    ages_M,
    qx_lower_M,
    qx_upper_M,
    color="lightgreen",
    alpha=0.35,
    label="Selang Mortalitas 95%"
)

plt.plot(
    ages_M,
    qx_mean_M,
    color="blue",
    linewidth=2,
    label="Rata-rata Monte Carlo"
)

plt.plot(
    ages_M,
    qx_estimasi_M,
    color="green",
    linestyle="-",
    linewidth=1.8,
    label="Estimasi Heligman-Pollard"
)

plt.plot(
    df_GAM_M["Usia"],
    df_GAM_M["qx_GAM71"],
    color="black",
    linestyle="--",
    linewidth=1.8,
    label="GAM 71 Male"
)

plt.xlabel("Usia (tahun)")
plt.ylabel("Probabilitas Kematian (qx)")
plt.title("Selang Mortalitas Male - Monte Carlo")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# 14. GRAFIK SELANG MORTALITAS LOGARITMIK
# ============================================================

plt.figure(figsize=(12, 5))

plt.fill_between(
    ages_M,
    np.maximum(qx_lower_M, 1e-20),
    np.maximum(qx_upper_M, 1e-20),
    color="lightgreen",
    alpha=0.35,
    label="Selang Mortalitas 95%"
)

plt.semilogy(
    ages_M,
    np.maximum(qx_mean_M, 1e-20),
    color="blue",
    linewidth=2,
    label="Rata-rata Monte Carlo"
)

plt.semilogy(
    ages_M,
    np.maximum(qx_estimasi_M, 1e-20),
    color="green",
    linestyle="-",
    linewidth=1.8,
    label="Estimasi Heligman-Pollard"
)

plt.semilogy(
    df_GAM_M["Usia"],
    np.maximum(
        df_GAM_M["qx_GAM71"], 1e-20
    ),
    color="black",
    linestyle="--",
    linewidth=1.8,
    label="GAM 71 Male"
)

plt.xlabel("Usia (tahun)")
plt.ylabel("Probabilitas Kematian (qx)")
plt.title("Selang Mortalitas Male - Skala Logaritmik")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()


# ============================================================
# 15. SIMPAN HANYA SATU FILE EXCEL
# ============================================================

output_file_M = "Selang_Mortalitas_Male.xlsx"

df_selang_M.to_excel(
    output_file_M,
    index=False
)

print("\n" + "=" * 60)
print("HASIL MONTE CARLO MALE TERSIMPAN")
print("=" * 60)

print("File:", output_file_M)
print("Jumlah usia:", len(df_selang_M))
print("Jumlah simulasi valid:", f"{N_SIM_M:,}")

print(
    "Kolom:",
    list(df_selang_M.columns)
)

print(
    "\nCatatan: Batas selang merupakan aproksimasi "
    "persentil histogram logaritmik."
)
